In [ ]:
## IBM smoke test — VQE-shaped path + market data

This matches **`hardware_smoke_test`** in `services/ibm_quantum.py`:

1. **`load_market_payload`** — tickers (yfinance) or explicit `returns` / `covariance` (same as portfolio API).
2. **`EfficientSU2(n, reps=1, entanglement="linear")`** — same ansatz family as IBM VQE in `methods/vqe.py`.
3. Fixed parameters (`zeros`), **SamplerV2** on IBM Runtime, marginal |1⟩ from counts → weights → **Sharpe** \( \mu^\top w / \sqrt{w^\top \Sigma w} \).

Production entrypoint: **`POST /api/config/ibm-quantum/smoke-test`** (optional `tickers`, `start_date`, `end_date`). When `tickers` is omitted, the API uses **Mag 7 + JPM** (see `_SMOKE_DEFAULT_TICKERS` in `services/ibm_quantum.py`).


In [ ]:
# Snippet: same market + weight logic as the API smoke test (no IBM job — numpy only)
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().resolve()
if (ROOT / "services").is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "services").is_dir():
    sys.path.insert(0, str(ROOT.parent))

from services.data_provider import load_market_payload
from services.ibm_quantum import _marginal_weights_from_counts

payload = {
    "returns": [0.10, 0.08],
    "covariance": [[0.04, 0.01], [0.01, 0.05]],
    "asset_names": ["AAA", "BBB"],
}
m = load_market_payload(payload)
mu, Sigma = m.returns, m.covariance
counts = {"00": 40, "01": 10, "10": 10, "11": 40}
w = _marginal_weights_from_counts(counts, len(mu), 0.001, 0.30)
port_ret = float(w @ mu)
vol = float(np.sqrt(max(w @ Sigma @ w, 0.0)))
sharpe = port_ret / vol if vol > 1e-10 else None
print("tickers:", m.tickers, "source:", m.source)
print("weights:", w, "Sharpe:", sharpe)
